# Tutorial 1: Data Preparation

This tutorial covers loading, preprocessing, and saving multimodal data for deepSCENIC.

## Setup

First, let's import the required packages:

In [1]:
import anndata as ad
import scanpy as sc

import deepscenic as ds

print(f"deepSCENIC version: {ds.__version__}")

deepSCENIC version: 0.1.0


## Download Sample Data

For this tutorial, we'll use the scenicplus MM_lines dataset (melanoma cell lines).

```{note}
Preprocessed sample datasets are coming soon. For now, provide your own scATAC and scRNA datasets to follow the tutorial.  
```

In [2]:
# For now, set your data directory manually:
import os

data_dir = "../../../../data/deepSCENIC/MM_lines/"
assert os.path.exists(data_dir), "dir not found"

## scRNA and scATAC Preprocessing

deepSCENIC combines RNA expression data, chromatin accessibility data, and genomic sequences to learn gene regulatory networks at the single-cell level. deepSCENIC follows the `scverse` convention for data formatting, and therefore we expect that your data is already preprocessed into the `anndata.AnnData` format. We will further preprocess these data modalities separately and later combine them into one `mudata.MuData` object, which is the format that deepSCENIC uses to run both training and inference.

### scRNA-seq Preprocessing

Load the gene expression matrix:

In [3]:
# Load scRNA-seq data
adata_rna = ad.read_h5ad(f"{data_dir}/raw_exprMat.h5ad")
print(f"RNA data: {adata_rna.n_obs} cells x {adata_rna.n_vars} genes")

RNA data: 936 cells x 12188 genes


We perform some basic filtering and normalization using standard `scanpy` functionality.

In [4]:
# Standard scRNA-seq preprocessing with scanpy
# (deepSCENIC expects log-normalized RNA data)
sc.pp.filter_genes(adata_rna, min_cells=10)
sc.pp.normalize_total(adata_rna)
sc.pp.log1p(adata_rna)

Additionally, we remove zero-variance genes (if there are any). These are genes that have the same constant expression across all cells (e.g. 0), which means they'll likely be completely uninformative for training.

In [5]:
ds.pp.remove_zero_variance_genes(adata_rna)

#### Fetch External Resources

Before further preprocessing, we need to fetch transcription factor lists and gene annotations from external databases.

In [6]:
# Fetch TF list from SCENIC+ resources
tfs = ds.fetch_tf_collection(species="human")  # also available: "mouse", "fly"
print(f"Found {len(tfs)} transcription factors")
tfs[:10]

Found 1892 transcription factors


['ZNF354C',
 'KLF12',
 'ZNF143',
 'ZIC2',
 'ZNF274',
 'SP2',
 'ZBTB7A',
 'BCL6B',
 'ZBTB49',
 'ZIC1']

In [7]:
# Fetch gene annotations from Ensembl
annot, chromsizes = ds.fetch_gene_annotation(
    species="hsapiens",
    biomart_host = "http://www.ensembl.org", # default
    transcript_type="protein_coding",
)
print(f"Annotations for {len(annot)} genes")
annot.head()

deepscenic.datasets - INFO - Loading cached gene annotation from /home/luna.kuleuven.be/u0166574/.cache/deepscenic/gene_annot_hsapiens_befba9cb_ucsc_protein_coding.parquet
deepscenic.datasets - INFO - Loaded annotation for 19354 genes from cache


Annotations for 19354 genes


,Chromosome,Start,End,Strand,Transcription_Start_Site,Transcript_type
Gene,,,,,,
MT-ND1,chrM,3307,4262,+,3307,protein_coding
MT-ND2,chrM,4470,5511,+,4470,protein_coding
MT-CO1,chrM,5904,7445,+,5904,protein_coding
MT-CO2,chrM,7586,8269,+,7586,protein_coding
MT-ATP8,chrM,8366,8572,+,8366,protein_coding


#### Mark Transcription Factors

deepSCENIC models gene regulation through transcription factors (TFs). We need to identify which genes in our dataset are TFs so the model knows which genes can act as regulators. This adds an `is_tf` column to `adata_rna.var` and stores the TF order in `adata_rna.uns['tf_order']`.

In [8]:
# Mark TFs in the RNA data
ds.pp.mark_tfs(adata_rna, tfs)
print(f"Marked {adata_rna.var['is_tf'].sum()} TFs in the dataset")

Marked 1462 TFs in the dataset


#### Add Gene Annotations

Gene annotations (chromosome positions) are needed for two purposes:
1. Computing the region-to-gene search space (which regions can regulate which genes)
2. Splitting features by chromosome for model evaluation

This adds `chromosome` and `tss` (transcription start site) columns to `adata_rna.var`.

In [9]:
# Add gene annotations (chromosome, TSS)
ds.pp.add_gene_annotation(adata_rna, annot)
adata_rna.var[['chromosome', 'tss', 'is_tf']].head()

deepscenic.pp - INFO - Added ['chromosome', 'tss'] to 12140 / 12188 genes
deepscenic.pp - INFO -   (48 genes not found in annotation)


,chromosome,tss,is_tf
ABCF2,chr7,151227205,True
ABL1,chr9,130713043,True
ACAA1,chr3,38137242,True
ACO1,chr9,32384603,True
ADARB1,chr21,45074580,True


### scATAC-seq Preprocessing

Load the chromatin accessibility matrix. This is required for training a deepSCENIC model, but not required for running inference or gene perturbation analyses using an already-trained model.

```{note}
We strongly recommend using **imputed** accessibility values (instead of raw) from [pycisTopic](https://pycistopic.readthedocs.io/en/latest/). The imputed matrix provides continuous accessibility values that are better suited for the VAE architecture than sparse binary fragment counts.
```

In [10]:
# Load scATAC-seq data (imputed accessibility from pycisTopic)
adata_atac = sc.read_h5ad(f"{data_dir}/fragment_matrix.h5ad")
print(f"ATAC data: {adata_atac.n_obs} cells x {adata_atac.n_vars} regions")

ATAC data: 936 cells x 281820 regions


#### Mark Differentially Accessible Regions (Optional)

Differentially Accessible Regions (DARs) are cell-type-specific regulatory regions identified through differential accessibility analysis. Marking DARs allows the model to upweight these biologically important regions during training.

```{note}
This step is **optional**. If you have DAR results from (for example) pycisTopic, you can load them here. Otherwise, skip this step.
```

In [11]:
# Optional: Mark DARs if you have pycisTopic differential accessibility results
# ds.pp.mark_dars(adata_atac, dar_dir="path/to/DARs/")

# Alternatively, provide a dictionary mapping cell types to region lists:
# dar_dict = {"CellType1": ["chr1:1000-2000", ...], "CellType2": [...]}
# ds.pp.mark_dars(adata_atac, dar_dict=dar_dict)

## Create MuData

deepSCENIC uses MuData to store multimodal data in a single object. The {func}`~deepscenic.pp.create_mudata` function combines the RNA and ATAC modalities and automatically parses region coordinates from the ATAC `var_names`. We expect the same cells in both data modalities. 

```{note}
The RNA and ATAC datasets must have identical cell barcodes (matching `obs_names`).
```

In [12]:
# Create MuData from RNA and ATAC
mdata = ds.pp.create_mudata(rna=adata_rna, atac=adata_atac)
mdata

MuData object with n_obs × n_vars = 936 × 294008
  2 modalities
    rna:	936 x 12188
      var:	'n_cells', 'is_tf', 'chromosome', 'tss'
      uns:	'log1p'
      layers:	'log_norm'
    atac:	936 x 281820
      var:	'chromosome', 'start', 'end'

## MuData Processing

Now we apply the preprocessing steps that operate on the combined MuData object.

### Compute Region-to-Gene Search Space

The region-to-gene (R2G) search space defines which genomic regions can potentially regulate which genes. This is based on genomic distance: regions closer to a gene's transcription start site (TSS) are more likely to regulate that gene.

The function computes a distance-weighted penalty matrix where:
- `max_distance`: Maximum distance from TSS to consider (default: 1Mb)
- `sigma`: Gaussian decay parameter (default: 100kb) - regions closer to TSS get lower penalty

**Important** Since we need to know the TSS per gene to deduce region-gene links we filter out genes without TSS annotation in this step. 

In [13]:
# Compute r2g penalty matrix (distance-based)
ds.pp.compute_r2g_penalty(
    mdata,
    max_distance=1_000_000,  # 1Mb
    sigma=100_000,           # 100kb Gaussian sigma for decay penalty
)

Computing R2G matrix: 100%|████████████████████████| 24/24 [00:10<00:00,  2.25it/s]
deepscenic.pp.search_space - INFO - 16 TFs without annotation will have 0 links
deepscenic.pp.search_space - INFO - Removed 32 non-TF genes without annotation from RNA modality
deepscenic.pp.search_space - INFO - Reordered RNA modality: 1462 TFs first, then 10694 other genes


### Build PPI Network (Optional)

The Protein-Protein Interaction (PPI) network models post-transcriptional regulation of TF activity. When TFs interact with other proteins, their regulatory activity can be modulated. This is captured through a Graph Attention Network (GAT) that learns from PPI data.

```{note}
This step is **optional**. If you don't have PPI data, the model will skip the PPI modulation component.
```

In [14]:
# Optional: Build PPI network for TF activity modulation
# Download STRING PPI data from https://string-db.org/
#
# ppi_df = ds.pp.load_string_ppi("path/to/string_links.txt", score_threshold=400)
# ds.pp.build_ppi_network(mdata, ppi_df, species="mouse")

### Split Data for Training and Evaluation

We split the data into training and test sets at two levels:

1. **Cell split**: Random split of cells (e.g., 80% train, 20% test)
2. **Feature split**: Chromosome-based split of genes/regions

```{important}
The feature split enables evaluation on held-out chromosomes to assess generalization to unseen genomic regions. Only the **E2 matrix** (region→gene weights) is trained on ALL genes and ALL regions. The rest of the parameters (encoder, decoders, TF2rNet) are trained on **TRAIN genes and TRAIN regions** only.
```

#### Transcription Factor Handling

By default, `split_features_by_chromosome()` uses `keep_tfs_in_both=True`, which marks all transcription factors (TFs) as available in both train and test splits. This is important because:

- TF expression is input to the encoder (needs all TFs available)
- TF→region links (E1 matrix) predict TF binding across all regions
- Only non-TF genes are strictly split by chromosome for evaluation

#### Split Structure

The split creates a 2×2 matrix for each modality:

```
                        GENES (by TSS chromosome)
                    ┌─────────────┬─────────────┐
                    │   TRAIN     │    TEST     │
                    │ (other chr) │ (chr7,11,   │
                    │             │  18,19)     │
     ┌──────────────┼─────────────┼─────────────┤
     │    TRAIN     │ rna_train   │ rna_E2      │
CELLS│   (80%)      │ [MAIN]      │ [E2 EVAL]   │
     ├──────────────┼─────────────┼─────────────┤
     │    TEST      │ train_eval  │ rna_test    │
     │   (20%)      │ [RECON EVAL]│ [HELD OUT]  │
     └──────────────┴─────────────┴─────────────┘
```

**Training uses:**
- Reconstruction loss computed on TRAIN genes and TRAIN regions only
- E2 sparsity loss applied to ALL region→gene links (including test genes)

**Evaluation uses:**
- `test cells × train features`: Reconstruction evaluation (same features, new cells)
- `train cells × test features`: E2 matrix evaluation (region→gene weights for held-out chromosomes)
- `test cells × test features`: Full held-out evaluation (new cells, new features)

In [15]:
# Split cells (80% train, 20% test)
ds.pp.split_cells(mdata, test_fraction=0.2, seed=42)
print(f"Cell split: {(mdata.obs['split'] == 'train').sum()} train, {(mdata.obs['split'] == 'test').sum()} test")

deepscenic.pp - INFO - Split cells: 748 train, 188 test


Cell split: 748 train, 188 test


In [16]:
# Split features by chromosome
ds.pp.split_features_by_chromosome(
    mdata,
    test_chromosomes=["chr16"],
)
print(f"Gene split: {(mdata['rna'].var['split'] == 'train').sum()} train, {(mdata['rna'].var['split'] == 'test').sum()} test")
print(f"Region split: {(mdata['atac'].var['split'] == 'train').sum()} train, {(mdata['atac'].var['split'] == 'test').sum()} test")

deepscenic.pp - INFO - 1462 TFs assigned to 'both' splits
deepscenic.pp - INFO - 16 genes without chromosome annotation assigned to 'train' split
deepscenic.pp - INFO - Split features by chromosome: 8275 ATAC regions, 465 genes in test set
deepscenic.pp - INFO -   (1462 TFs assigned to 'both' splits)


Gene split: 10229 train, 465 test
Region split: 273545 train, 8275 test


## Saving and Loading

deepSCENIC stores all preprocessed data in a single `.h5mu` file. This includes both modalities, all annotations, the R2G matrix, and split information.

### Save Preprocessed Data

Use {func}`~deepscenic.write` to save the preprocessed MuData. The function validates the data against the deepSCENIC schema before saving to ensure all required fields are present.

In [17]:
# Save preprocessed MuData
ds.write(mdata, data_dir + "preprocessed_data.h5mu")

deepscenic.io - INFO - Wrote ../../../../data/deepSCENIC/MM_lines/preprocessed_data.h5mu: 936 cells


### Load Preprocessed Data

In future sessions, use {func}`~deepscenic.read` to load your preprocessed data. The function validates the schema on load to ensure data integrity.

In [21]:
# Load preprocessed MuData in future sessions
# mdata = ds.read(data_dir + "preprocessed_data.h5mu")

## Next Steps

In the next tutorial, we'll train the deepSCENIC model:

- {doc}`02_training`